# Chroma VectorStore

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 벡터 저장소(VectorStore)의 역할과 Chroma의 특징을 설명할 수 있어요
- 문서를 Chroma에 저장하고 유사도 검색을 수행할 수 있어요
- 메타데이터 필터링, MMR 검색, Retriever 변환을 활용할 수 있어요
- RAG 체인에 Chroma를 통합할 수 있어요

## 사전 지식

- 6-3 Embeddings 학습 완료 (임베딩 개념 이해 필수)
- OpenAI API 키 설정

---

## VectorStore란?

벡터 저장소(VectorStore)는 임베딩 벡터를 저장하고 유사도 기반 검색을 제공하는 데이터베이스예요. RAG 시스템에서 "문서 창고" 역할을 해요.

```mermaid
flowchart LR
    subgraph 저장["저장 단계 (색인)"]
        D["문서들"]:::input
        E["임베딩 모델"]:::process
        VS["VectorStore<br/>(Chroma)"]:::storage
        D --> E --> VS
    end

    subgraph 검색["검색 단계 (쿼리)"]
        Q["사용자 질문"]:::input
        QE["임베딩 모델"]:::process
        QV["질문 벡터"]:::process
        R["유사 문서<br/>Top-K"]:::output
        Q --> QE --> QV
        QV -->|"유사도 비교"| VS
        VS --> R
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## Chroma 특징

- 로컬 개발에 최적화된 오픈소스 벡터 DB
- `persist_directory`로 디스크에 영구 저장
- 메타데이터 필터링으로 정밀 검색
- LangChain과 원활하게 통합

**참고 문서**: [Chroma 공식 문서](https://docs.trychroma.com/) | [LangChain Chroma 통합](https://python.langchain.com/docs/integrations/vectorstores/chroma/)

## 환경 설정

In [1]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

---

## 1. 데이터 준비

벡터 저장소에 넣을 문서를 로드하고 분할해볼게요. NLP 키워드 문서와 금융 키워드 문서를 사용해요.

In [2]:

# ---------------------------------------------------
# 문서 로드 및 분할 (벡터 저장소에 넣을 데이터 준비)
# ---------------------------------------------------

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할 설정 (chunk_size: 600자, overlap: 없음)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=0
)

# 1단계: 문서 로더 생성
loader_nlp = TextLoader("./data/nlp-keywords.txt")
loader_finance = TextLoader("./data/finance-keywords.txt")

# 2단계: 문서 로드 및 분할
# load_and_split(): 로드 + 분할을 한 번에 처리하는 편의 메서드
docs_nlp = loader_nlp.load_and_split(text_splitter)
docs_finance = loader_finance.load_and_split(text_splitter)

# 3단계: 문서 개수 확인
print(f"NLP: {len(docs_nlp)}")
print(f"Finance: {len(docs_finance)}")


NLP: 11
Finance: 6


---

## 2. VectorStore 생성하기

### 2.1 from_documents로 생성

`from_documents()`는 Document 객체 리스트를 받아서 임베딩 후 저장소를 만들어주는 가장 편리한 방법이에요.

**주요 파라미터**:
- `documents`: Document 객체 리스트
- `embedding`: 임베딩 모델 (`OpenAIEmbeddings` 등)
- `collection_name`: 컬렉션 이름 (네임스페이스 역할)
- `persist_directory`: 디스크 저장 경로 (지정 안 하면 메모리에만 저장)

In [3]:
# ---------------------------------------------------
# Chroma VectorStore 생성 (메모리)
# ---------------------------------------------------

# ============================================================
# TODO: from_documents()로 Chroma 벡터 저장소를 생성해보세요
# 힌트: Chroma.from_documents(documents, embedding, collection_name=...)
# 예상 결과: "VectorStore가 메모리에 생성되었습니다." 출력
# ============================================================

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1단계: Chroma.from_documents()로 메모리 기반 VectorStore 생성
# Chroma: 로컬 개발에 최적화된 오픈소스 벡터 DB
# collection_name: 같은 DB 안에서 데이터를 구분하는 네임스페이스
vectorstore = Chroma.from_documents(
    documents=docs_nlp,
    embedding=OpenAIEmbeddings(),
    collection_name="tech_keywords"
)



Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### 2.2 디스크에 영구 저장하기

`persist_directory`를 지정하면 프로세스를 재시작해도 데이터가 유지돼요.

> **실무 팁**: 운영 환경에서는 항상 `persist_directory`를 지정해서 저장해두세요. 임베딩 API 비용이 다시 발생하지 않아요.

In [4]:
# ---------------------------------------------------
# Chroma VectorStore 생성 (디스크 영구 저장)
# ---------------------------------------------------

# ============================================================
# TODO: persist_directory를 지정해 디스크에 저장되는 VectorStore를 만들어보세요
# 힌트: Chroma.from_documents(..., persist_directory="./chroma_tech_db")
# 예상 결과: 지정 경로에 폴더가 생성되고 "저장되었습니다." 출력
# ============================================================

# 1단계: 디스크 저장 경로 지정
DB_PATH = "./chroma_tech_db"

# 2단계: persist_directory로 디스크에 영구 저장
# persist_directory: 지정 시 프로세스 재시작 후에도 데이터 유지

vectorstore = Chroma.from_documents(
    documents=docs_nlp,
    embedding=OpenAIEmbeddings(),
    collection_name="tech_keywords",
    persist_directory=DB_PATH
)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### 2.3 저장된 VectorStore 로드하기

이미 저장된 벡터 저장소를 다시 불러올 수 있어요. 임베딩을 새로 계산하지 않아도 돼요.

In [5]:

# ---------------------------------------------------
# 디스크에 저장된 Chroma VectorStore 로드
# ---------------------------------------------------

# 1단계: 저장된 VectorStore 로드 (임베딩 재계산 불필요)
# Chroma(): from_documents() 대신 로드 전용 생성자 사용
loaded_vectorstore = Chroma(
    persist_directory=DB_PATH,
    embedding_function=OpenAIEmbeddings(),
    collection_name="tech_keywords"
)

# 2단계: 저장된 데이터 개수 확인
stored_data = loaded_vectorstore.get()
stored_data

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['e65cddef-d050-4598-8edb-e06cfb4c43b7',
  '52ceec13-a2fd-458e-851c-af8712c352d0',
  'fb0967a6-40c3-4ab3-ac8e-135119d5482b',
  '7ab3f04c-3b48-449a-b1a4-3e3c481b5768',
  'b61a4698-4d60-48f7-a44c-535e90fa4901',
  '3018f2a8-3b16-4019-b780-7d0f7c2750df',
  'd7f6f1b6-62a9-4748-b81b-5dc7e22df54c',
  '2e33a38f-65fc-40aa-ad6d-9a6d6a89a0dd',
  'bf463b3c-b443-4a3e-99b5-b089021b8d1d',
  'f55c25aa-c993-4d05-9678-0e574f0d9669',
  '243ba11b-e32f-48fd-a8a5-1190bae01f8c',
  '85a0c380-3a91-4547-adab-49ac131497d1',
  'a9debdfd-62f9-4e33-b673-f388f30fd965',
  '9fd9f30a-2b2f-4506-aa12-2af4e1ad0af2',
  '8cf992e9-5964-4190-8bad-8c4b9ee10a17',
  '03da62d7-6694-438c-9bcc-11a160b512e9',
  'fe71c531-b089-4e19-8612-11adab94f80c',
  '2d89c4d8-cb37-4428-a574-cedbd342fb27',
  'd09d6006-13ca-4869-91e1-1a8991a37939',
  '7ef8ede9-6751-440d-806b-e88cb1a7a756',
  'f8690cca-7077-41e1-8ec9-42f7faeb3a4d',
  '2f085bdd-1335-4417-8a0d-951b1a2f7865'],
 'embeddings': None,
 'documents': ['Semantic Search\n\n정의: 의미론적 검색은

---

## 3. 유사도 검색 (Similarity Search)

### 3.1 기본 유사도 검색

`similarity_search()`는 쿼리와 가장 유사한 문서 k개를 반환해요. 내부적으로 코사인 유사도를 계산해서 순위를 매겨요.

In [6]:
# ---------------------------------------------------
# 기본 유사도 검색 (similarity_search)
# ---------------------------------------------------

# ============================================================
# TODO: similarity_search()로 쿼리와 유사한 문서를 검색해보세요
# 힌트: loaded_vectorstore.similarity_search("쿼리 텍스트")
# 예상 결과: 기본 4개 문서가 유사도 순으로 반환
# ============================================================

# 1단계: 기본 검색 실행 (k=4, 기본값)
# similarity_search(): 쿼리를 자동으로 임베딩 후 유사 문서 k개 반환
results = loaded_vectorstore.similarity_search("임베딩과 벡터 검색에 대해 알려줘")

results

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='b61a4698-4d60-48f7-a44c-535e90fa4901', metadata={'source': './data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='03da62d7-6694-438c-9bcc-11a160b512e9', metadata={'source': './data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미

### 3.2 검색 결과 개수 조정

`k` 파라미터로 반환받을 문서 개수를 조절해요.

In [7]:

# ---------------------------------------------------
# k 파라미터로 검색 결과 개수 조정
# ---------------------------------------------------

# k=2: 상위 2개 문서만 반환

results_top2 = loaded_vectorstore.similarity_search("임베딩과 벡터 검색에 대해 알려줘", k=2)

results_top2


[Document(id='b61a4698-4d60-48f7-a44c-535e90fa4901', metadata={'source': './data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='03da62d7-6694-438c-9bcc-11a160b512e9', metadata={'source': './data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미

### 3.3 메타데이터 필터링

`filter` 파라미터로 메타데이터 조건에 맞는 문서만 골라서 검색할 수 있어요. 특정 출처나 카테고리의 문서만 검색할 때 유용해요.

> **실무 팁**: 여러 종류의 문서를 하나의 VectorStore에 저장할 때는 `source`, `category`, `date` 같은 메타데이터를 잘 설계해두면 나중에 원하는 범위만 검색할 수 있어요.

In [8]:
# ---------------------------------------------------
# 메타데이터 필터로 특정 출처 문서만 검색
# ---------------------------------------------------

# ============================================================
# TODO: filter 파라미터로 특정 source의 문서만 검색해보세요
# 힌트: similarity_search(..., filter={"source": "경로"})
# 예상 결과: nlp-keywords.txt 출처 문서만 반환
# ============================================================

vectorstore = Chroma.from_documents(
    documents= docs_nlp + docs_finance,
    embedding=OpenAIEmbeddings(),
    collection_name="keywords"
)

# 1단계: filter로 메타데이터 조건 지정
# filter: Chroma에서 메타데이터 조건으로 검색 범위를 좁힘
query = "데이터 분석과 예측 모델과 주식에서의 영향력"

unfiltered = vectorstore.similarity_search(query, k=3)

print(f' ==> [Line 22]: \033[38;2;0;52;23m[unfiltered]\033[0m({type(unfiltered).__name__}) = \033[38;2;40;129;80m{unfiltered}\033[0m')

# 2단계: 메타데이터로 필터링
# NLP 문서
nlp_filtered = vectorstore.similarity_search(
    query, k=3,
    filter={"source": "./data/nlp-keywords.txt"}
)

print(f' ==> [Line 31]: \033[38;2;89;92;148m[nlp_filtered]\033[0m({type(nlp_filtered).__name__}) = \033[38;2;139;83;143m{nlp_filtered}\033[0m')



Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


 ==> [Line 22]: [unfiltered](list) = [Document(id='6acae8b0-9ae6-4b94-9e71-8775a3b25795', metadata={'source': './data/finance-keywords.txt'}, page_content='정의: 시장 가중치는 특정 기업이나 섹터가 전체 지수에서 차지하는 비중을 나타냅니다.\n예시: 기술 섹터는 S&P 500 지수에서 가장 큰 시장 가중치를 차지하고 있습니다.\n연관키워드: 포트폴리오 구성, 섹터 분석, 자산 배분\n\nGrowth Stock\n\n정의: 성장주는 평균 이상의 높은 성장률을 보이는 기업의 주식을 의미합니다.\n예시: 페이스북(메타)과 같은 기술 기업들은 S&P 500에 포함된 대표적인 성장주로 꼽힙니다.\n연관키워드: 고성장 기업, 기술주, 투자 전략\n\nValue Stock\n\n정의: 가치주는 현재 시장 가치가 내재 가치보다 낮다고 평가되는 기업의 주식을 말합니다.\n예시: 워렌 버핏이 투자한 코카콜라는 S&P 500에 포함된 대표적인 가치주 중 하나입니다.\n연관키워드: 가치 투자, 배당주, 안정적 수익\n\nMarket Volatility\n\n정의: 시장 변동성은 주식 시장의 가격 변동 폭을 나타내는 지표입니다.\n예시: VIX 지수(변동성 지수)가 상승하면 S&P 500 지수의 변동성이 높아질 것으로 예상됩니다.\n연관키워드: 리스크 관리, 투자 심리, 헤지 전략\n\nEquity Research'), Document(id='6ff0f9cc-dfe7-42d2-bf5e-667eb81534e7', metadata={'source': './data/finance-keywords.txt'}, page_content='정의: 주식 리서치는 기업의 재무 상태, 사업 모델, 경쟁력 등을 분석하여 투자 의사 결정을 돕는 활동입니다.\n예시: 골드만삭스의 애널리스트들이 S&P 500 기업들에 대한 분기별 실적 전망을 발표했습니다.\n연관키워드: 투자 분석, 기

---

## 4. 문서 추가 및 관리

### 4.1 문서 추가

기존 VectorStore에 새 문서를 추가할 수 있어요. 전체를 다시 만들 필요 없어요.

In [9]:
# ---------------------------------------------------
# 기존 VectorStore에 새 문서 추가
# ---------------------------------------------------

from langchain_core.documents import Document

# 1단계: 새 문서 추가
# add_documents(): 기존 저장소를 유지하면서 새 문서를 덧붙임
new_docs = [
    Document(
        page_content="Multi Head attention은 트랜스포머의 핵심 알고리즘입니다.",
        metadata={"source": "custom_data", "topic": "attention"}
    ),
    Document(
        page_content="Multi Head attention은 트랜스포머의 핵심 알고리즘입니다.",
        metadata={"source": "custom_data", "topic": "attention"}
    ),
]

new_doc_ids = vectorstore.add_documents(new_docs)
print(f' ==> [Line 20]: \033[38;2;41;93;32m[new_doc_ids]\033[0m({type(new_doc_ids).__name__}) = \033[38;2;97;50;175m{new_doc_ids}\033[0m')




 ==> [Line 20]: [new_doc_ids](list) = ['b35e93a3-5865-4ea9-abfb-71ff186653fb', '8abcd029-7d83-4f7c-93b2-aa9eabf93bfc']


---

## 5. Retriever로 변환

### 5.1 기본 Retriever

`as_retriever()`로 VectorStore를 Retriever(리트리버)로 변환하면 LangChain 체인에서 표준 인터페이스로 사용할 수 있어요.

In [10]:
# ---------------------------------------------------
# VectorStore를 Retriever로 변환 (체인 통합용)
# ---------------------------------------------------

# ============================================================
# TODO: as_retriever()로 VectorStore를 Retriever로 변환해보세요
# 힌트: loaded_vectorstore.as_retriever()
# 예상 결과: 검색된 문서 4개 출력
# ============================================================

# 1단계: Retriever 생성
# as_retriever(): VectorStore를 LangChain 표준 검색 인터페이스로 변환
# Retriever는 invoke() 메서드를 지원하여 체인에 바로 연결 가능
retriever = vectorstore.as_retriever()

# 2단계: 검색 실행
retrieved_docs = retriever.invoke("GPT와 대규모 언어 모델에 대해 설명해줘.")
print(f' ==> [Line 17]: \033[38;2;26;145;106m[retrieved_docs]\033[0m({type(retrieved_docs).__name__}) = \033[38;2;186;15;50m{retrieved_docs}\033[0m')




 ==> [Line 17]: [retrieved_docs](list) = [Document(id='6cd0e870-aa59-4af9-bd61-f29a8fbda292', metadata={'source': './data/nlp-keywords.txt'}, page_content='GPT (Generative Pretrained Transformer)\n\n정의: GPT는 대규모의 데이터셋으로 사전 훈련된 생성적 언어 모델로, 다양한 텍스트 기반 작업에 활용됩니다. 이는 입력된 텍스트에 기반하여 자연스러운 언어를 생성할 수 있습니다.\n예시: 사용자가 제공한 질문에 대해 자세한 답변을 생성하는 챗봇은 GPT 모델을 사용할 수 있습니다.\n연관키워드: 자연어 처리, 텍스트 생성, 딥러닝\n\nInstructGPT\n\n정의: InstructGPT는 사용자의 지시에 따라 특정한 작업을 수행하기 위해 최적화된 GPT 모델입니다. 이 모델은 보다 정확하고 관련성 높은 결과를 생성하도록 설계되었습니다.\n예시: 사용자가 "이메일 초안 작성"과 같은 특정 지시를 제공하면, InstructGPT는 관련 내용을 기반으로 이메일을 작성합니다.\n연관키워드: 인공지능, 자연어 이해, 명령 기반 처리\n\nKeyword Search'), Document(id='61ee03df-d5e6-482e-bedb-f04b117b060a', metadata={'source': './data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다

### 5.2 MMR (Maximum Marginal Relevance) 검색

MMR(최대 한계 관련성)은 유사도와 다양성을 동시에 고려하는 검색 방법이에요. 비슷한 문서가 중복으로 나오는 문제를 줄여줘요.

**주요 파라미터**:
- `k`: 최종 반환 문서 개수
- `fetch_k`: 후보 문서 개수 (이 중에서 k개 선택)
- `lambda_mult`: 0이면 최대 다양성, 1이면 최대 유사도

> **실무 팁**: 검색 결과가 너무 비슷한 문서들로만 채워진다면 MMR을 적용해보세요. `lambda_mult=0.5` 정도로 시작해서 조절해보세요.

In [13]:
# ---------------------------------------------------
# MMR 검색으로 다양성을 고려한 Retriever 설정
# ---------------------------------------------------

# ============================================================
# TODO: search_type="mmr"로 MMR 방식 Retriever를 설정해보세요
# 힌트: as_retriever(search_type="mmr", search_kwargs={"k": ..., "fetch_k": ..., "lambda_mult": ...})
# 예상 결과: 유사도만 고려했을 때보다 다양한 주제의 문서 반환
# ============================================================
query = "시장 분석과 데이터 처리 기법"
regular_results = vectorstore.similarity_search(query, k=4)
print(f' ==> [Line 11]: \033[38;2;221;221;184m[regular_results]\033[0m({type(regular_results).__name__}) = \033[38;2;183;84;129m{regular_results}\033[0m')


# 1단계: MMR Retriever 설정
# search_type="mmr": MMR(Maximum Marginal Relevance) 알고리즘 사용
# fetch_k: 유사도로 뽑을 후보 문서 수 (이 중에서 k개 선택)
# lambda_mult: 0 = 최대 다양성, 1 = 최대 유사도 (0.5가 균형)

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 10, "lambda_mult": 0.5}
)
mmr_results = mmr_retriever.invoke(query)
print(f' ==> [Line 24]: \033[38;2;17;199;13m[mmr_results]\033[0m({type(mmr_results).__name__}) = \033[38;2;205;254;50m{mmr_results}\033[0m')



 ==> [Line 11]: [regular_results](list) = [Document(id='6acae8b0-9ae6-4b94-9e71-8775a3b25795', metadata={'source': './data/finance-keywords.txt'}, page_content='정의: 시장 가중치는 특정 기업이나 섹터가 전체 지수에서 차지하는 비중을 나타냅니다.\n예시: 기술 섹터는 S&P 500 지수에서 가장 큰 시장 가중치를 차지하고 있습니다.\n연관키워드: 포트폴리오 구성, 섹터 분석, 자산 배분\n\nGrowth Stock\n\n정의: 성장주는 평균 이상의 높은 성장률을 보이는 기업의 주식을 의미합니다.\n예시: 페이스북(메타)과 같은 기술 기업들은 S&P 500에 포함된 대표적인 성장주로 꼽힙니다.\n연관키워드: 고성장 기업, 기술주, 투자 전략\n\nValue Stock\n\n정의: 가치주는 현재 시장 가치가 내재 가치보다 낮다고 평가되는 기업의 주식을 말합니다.\n예시: 워렌 버핏이 투자한 코카콜라는 S&P 500에 포함된 대표적인 가치주 중 하나입니다.\n연관키워드: 가치 투자, 배당주, 안정적 수익\n\nMarket Volatility\n\n정의: 시장 변동성은 주식 시장의 가격 변동 폭을 나타내는 지표입니다.\n예시: VIX 지수(변동성 지수)가 상승하면 S&P 500 지수의 변동성이 높아질 것으로 예상됩니다.\n연관키워드: 리스크 관리, 투자 심리, 헤지 전략\n\nEquity Research'), Document(id='b686d615-2e7b-485d-b31a-dbf04395cb9e', metadata={'source': './data/nlp-keywords.txt'}, page_content='정의: 토크나이저는 텍스트 데이터를 토큰으로 분할하는 도구입니다. 이는 자연어 처리에서 데이터를 전처리하는 데 사용됩니다.\n예시: "I love programming."이라는 문장을 ["I", "love", "programming", 

---

## 6. RAG 체인 구성

Retriever를 활용해서 간단한 RAG(검색 증강 생성) 체인을 만들어볼게요.

In [16]:
# ---------------------------------------------------
# Retriever를 활용한 RAG 체인 구성 및 실행
# ---------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1단계: 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    """
    다음 문서를 참고하여 질문에 답변하시오.
    참고할만한 문서가 없으면 범용적인 지식을 이용해 답변하세요.

    참고문서: {context}

    질문: {question}

    답변:
    """
)



# 2단계: 문서 포맷 함수
def format_docs(docs):
    """검색된 문서를 하나의 문자열로 결합"""
    return "\n\n".join(doc.page_content for doc in docs)

# 3단계: RAG 체인 구성
# RunnablePassthrough(): 입력(question)을 그대로 통과시키는 패스스루 컴포넌트
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

# 4단계: 질문 실행
# question = "트랜스포머와 어탠션 메커니즘에 대해 설명해줘."
question = "뜬금 없는 이상한 질문"

# 아래에 코드를 작성하세요
res = rag_chain.invoke(question)

res


'"뜬금 없는 이상한 질문"이라는 표현은 일반적으로 예상치 못한 질문이나 대화의 흐름과 서식에 어긋나는 질문을 의미합니다. 이러한 질문은 대화의 맥락을 고려하지 않거나, 특정 주제를 비틀어 놓는 것일 수 있습니다. 일반적으로 유머를 목적으로 하거나, 대화를 이완시키기 위해 등장하는 경우가 많습니다. 이런 질문에 대한 답변은 상황에 따라 유동적이기 때문에, 주어진 문서나 주제와 관련이 없더라도 일반적인 정보를 제공하는 것입니다. \n\n예를 들어, "만약 외계인이 지구에 온다면 어떤 주식을 사야 할까요?"와 같은 질문은 뜬금 없는 질문에 해당할 수 있습니다. 이에 대한 답변은 다음과 같이 할 수 있습니다: "외계인이 가장 흥미를 가질 만한 기술주나 우주 산업 관련 주식일 것입니다."'

---

## 마무리

### 핵심 요약

Chroma VectorStore의 주요 기능을 정리해볼게요:

| 기능 | 메서드 | 설명 |
|------|--------|------|
| 저장소 생성 | `from_documents()` | Document 리스트로 생성 |
| 영구 저장 | `persist_directory=` | 디스크에 저장 |
| 기본 검색 | `similarity_search()` | 유사도 기반 Top-K |
| 다양성 검색 | `as_retriever(search_type="mmr")` | MMR 알고리즘 |
| 필터 검색 | `filter={"key": "val"}` | 메타데이터 조건 |
| 문서 추가 | `add_documents()` | 기존 저장소에 추가 |
| Retriever | `as_retriever()` | 체인 통합 |

**Chroma는 이럴 때 사용해요**: 로컬 개발, 프로토타입 제작, 수천~수만 개 문서 처리, 재시작 후 데이터 유지가 필요한 소규모 프로젝트

### 다음 학습

**6-4 노트북 02**: FAISS — 대규모 데이터 고속 검색에 최적화된 벡터 저장소를 배워볼게요.